# Train / Test Creator — prototype run

Reads **`unified_schema_vcb.return_5day__final__d20_h5`** — the table
`final_features` materialised from the feature-selection runs — and writes
windowed train/val/test tensors under `src/train_test_set/`.

Nothing is selected, tuned or joined here. The table already holds exactly the
channels chosen upstream, so this notebook is **split → impute → scale →
window → save**, with every statistic computed on the train slice only.

> ⚠️ `d` and `h` are read from the TABLE NAME, never set below. See
> [CONTEXT.md](CONTEXT.md) §2 for why that used to be a free parameter and what
> it cost.

## Import

In [1]:
import os
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, os.path.abspath(".."))

from train_test_creator import TrainTestCreator

pd.set_option("display.max_columns", 50)

## Parameters

`train_ratio` / `val_ratio` cut the DATE axis; test takes the remainder.
`purge=False` reproduces the old un-purged split and is for comparison only —
never for a result.

In [2]:
TICKER = "vcb"
TABLE = "return_5day__final__d20_h5"

TRAIN_RATIO = 0.70
VAL_RATIO = 0.15

creator = TrainTestCreator(
    ticker=TICKER,
    table=TABLE,
    train_ratio=TRAIN_RATIO,
    val_ratio=VAL_RATIO,
    scale_target=True,
    on_untrainable="drop",
    purge=True,
)

print(f"source   {creator.schema_table}")
print(f"target   {creator.target}  (h={creator.horizon})")
print(f"lookback d={creator.lookback}  ->  purge gap {creator.purge_gap} samples")
print(f"dataset  {creator.name}")
print(f"output   {creator.output_dir()}")

source   unified_schema_vcb.return_5day__final__d20_h5
target   return_5day  (h=5)
lookback d=20  ->  purge gap 24 samples
dataset  vcb__return_5day__final__d20_h5__tr70_val15_test15__std
output   D:\GIT\master-thesis\src\train_test_set\vcb__return_5day__final__d20_h5__tr70_val15_test15__std


## Read the table

Through `UnifiedSchemaReader`, which casts from `information_schema`: psycopg2
returns `numeric` as `Decimal` and pandas carries that as dtype `object`, so a
raw read would hand `StandardScaler` seventeen object columns.

In [3]:
frame, comment = creator.read()

print(f"{frame.shape[0]} rows x {frame.shape[1]} columns")
print(f"{frame['date'].min().date()} -> {frame['date'].max().date()}")
print(f"tickers: {sorted(frame['ticker'].unique())}")
print()
print("COMMENT ON TABLE:")
print(comment[:600])

4235 rows x 207 columns
2009-06-30 -> 2026-06-25
tickers: ['VCB']

COMMENT ON TABLE:
Final feature table built by final_features from 19 feature-selection run(s) sharing target='return_5day' and setup [lookback_d=20, horizon_h=5, normalize=none, feature_normalize=not_set, max_features=12, corr_threshold=0.9, n_splits=5, min_train=500, random_state=42, selector_class=FeatureSelector]. 203 channels from 19 pool(s). Run evidence: no_null=19. ⚠️ evidence=no_null means no bar was computed for that run — a ranking without a null is descriptive, not evidence (feature_selection/CONTEXT.md §14b). Source runs: 2026-08-04_205945__vcb__basic__return_5day; 2026-08-07_014755__vcb__basic+eco


In [4]:
frame

,date,exchange,ticker,return_5day,avg_vol_per_buy_order,buy_order_vol,close_adjust,foreign_net_volume,foreign_own,foreign_room_left,foreign_sell_value,foreign_sell_volume,low,n_sell_orders,open,prop_buy_vol,value_negotiated,volume_negotiated,australia__economy__business__economics__aulei,australia__economy__consumer__economics__aublr,australia__economy__gdp__fred__naexkp01auq657s,australia__economy__government__economics__aufe,australia__economy__government__economics__augbv,australia__economy__housing__economics__aubp,australia__economy__labor__economics__auyur,...,united_kingdom__economy__labor__economics__gbreeb,united_kingdom__economy__money__economics__gboiar,united_kingdom__economy__prices__economics__gbppimm,united_kingdom__economy__trade__economics__gbed,usa__economy__business__economics__uscor,usa__economy__business__fred__ltotalnsa,usa__economy__business__fred__truckd11,usa__economy__gdp__fred__a191rl1q225sbea,usa__economy__government__fred__fyoint,usa__economy__housing__economics__ushor,usa__economy__housing__fred__boaaahorusq156n,usa__economy__money__economics__usfbi,usa__economy__prices__fred__will5000ind,usa__economy__prices__fred__wpu0851,usa__economy__prices__fred__wpusi019011,usa__economy__trade__fred__expjp,vietnam__economy__gdp__economics__vngdpa,vietnam__economy__money__economics__vnm1,vietnam__economy__prices__economics__vncir,vietnam__economy__prices__economics__vncirmm,vietnam__economy__prices__economics__vncpit,vietnam__economy__prices__economics__vnep,vietnam__economy__prices__economics__vnirmm,vietnam__economy__trade__economics__vnbot,vietnam__economy__trade__economics__vnexp
0,2009-06-30,HOSE,VCB,-0.058050,NaN,NaN,9130.0,NaN,NaN,NaN,NaN,NaN,60000.0,NaN,60000.0,NaN,0.000000e+00,0,-0.13,8.81,0.617159,2.765600e+10,-2.641000e+09,10883.0,12.3,...,0.2,0.3711,0.2,6.188636e+12,219.0,923902.0,76.4,-0.7,2.527570e+11,67.3,46.5,-2.257600e+10,35.41,182.1,309.2,4.270519e+09,NaN,4.429840e+14,NaN,NaN,NaN,124.8,0.43,-1.254000e+09,4.415000e+09
1,2009-07-01,HOSE,VCB,-0.082519,NaN,NaN,9210.0,NaN,NaN,NaN,NaN,NaN,59500.0,NaN,63000.0,NaN,2.124300e+11,3,-0.13,8.81,0.617159,2.765600e+10,-2.641000e+09,10883.0,12.3,...,0.2,0.4195,0.2,6.188636e+12,219.0,857425.0,75.3,-0.7,2.527570e+11,67.3,46.5,-2.257600e+10,35.60,181.6,321.1,3.992692e+09,NaN,4.429840e+14,NaN,NaN,NaN,124.8,0.43,-1.254000e+09,4.415000e+09
2,2009-07-02,HOSE,VCB,-0.069083,NaN,NaN,8830.0,NaN,NaN,NaN,NaN,NaN,57500.0,NaN,59500.0,NaN,2.310000e+09,0,-0.13,8.81,0.617159,2.765600e+10,-2.641000e+09,10883.0,12.3,...,0.2,0.4160,0.2,6.188636e+12,219.0,857425.0,75.3,-0.7,2.527570e+11,67.3,46.5,-2.257600e+10,34.57,181.6,321.1,3.992692e+09,NaN,4.429840e+14,NaN,NaN,NaN,124.8,0.43,-1.254000e+09,4.415000e+09
3,2009-07-03,HOSE,VCB,-0.080891,NaN,NaN,8530.0,NaN,NaN,NaN,NaN,NaN,56000.0,NaN,56500.0,NaN,2.541000e+09,0,-0.13,8.81,0.617159,2.765600e+10,-2.641000e+09,10883.0,12.3,...,0.2,0.4300,0.2,6.188636e+12,219.0,857425.0,75.3,-0.7,2.527570e+11,67.3,46.5,-2.257600e+10,34.57,181.6,321.1,3.992692e+09,NaN,4.429840e+14,NaN,NaN,NaN,124.8,0.43,-1.254000e+09,4.415000e+09
4,2009-07-06,HOSE,VCB,-0.162738,NaN,NaN,8910.0,NaN,NaN,NaN,NaN,NaN,56000.0,NaN,56000.0,NaN,0.000000e+00,0,-0.13,8.81,0.617159,2.765600e+10,-2.641000e+09,10883.0,12.3,...,0.2,0.4171,0.2,6.188636e+12,229.0,857425.0,75.3,-0.7,2.527570e+11,67.3,46.5,-2.257600e+10,34.58,181.6,321.1,3.992692e+09,NaN,4.429840e+14,NaN,NaN,NaN,124.8,0.43,-1.254000e+09,4.415000e+09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4230,2026-06-19,HOSE,VCB,NaN,1451.0,7831883.0,61700.0,462800.0,20.25,814590649.0,7.968278e+10,1294000.0,61200.0,4321.0,61900.0,752200.0,0.000000e+00,0,0.10,10.51,0.274335,6.307200e+10,1.015700e+10,17207.0,11.0,...,0.5,3.7300,1.5,8.472849e+12,433.0,1467381.0,114.3,1.5,9.700650e+11,65.3,45.4,5.050000e+10,NaN,257.0,774.3,8.962044e+09,2.805870e+14,4.318076e+15,4.66,0.88,105.

## Build

One call. It drops the unlabelled tail, cuts the date axis, purges `d + h - 1`
samples at each boundary, imputes with the **train median**, scales with a
**train-fit** `StandardScaler`, and stacks the windows per ticker.

In [5]:
data = creator.build(frame)

print(f"{data.rows_read} rows read, {data.rows_unlabelled} unlabelled tail dropped")
print(f"{data.n_features} features kept, {len(data.dropped_columns)} dropped")
for column, reason in data.dropped_columns.items():
    print(f"  [drop] {column}: {reason}")

4235 rows read, 5 unlabelled tail dropped
202 features kept, 1 dropped
  [drop] prop_buy_vol: constant across the train slice — imputation makes it a constant in training and it varies at test, so the model is handed a live signal it could never fit a response to


In [6]:
data.shapes()

,split,samples,lookback,features,first_date,last_date
0,train,2918,20,202,2009-07-27,2021-04-05
1,val,610,20,202,2021-05-13,2023-10-17
2,test,635,20,202,2023-11-21,2026-06-18


## ⚠️ Check the purge held

The point of the whole split: no train label period may reach into the first
window of the next split. `last train label + h` must be strictly less than
`next window opens`.

In [7]:
labelled = frame.dropna(subset=[creator.target]).reset_index(drop=True)
position = pd.Index(labelled["date"].dt.strftime("%Y-%m-%d"))

checks = []
for earlier, later in (("train", "val"), ("val", "test")):
    last_label = int(position.get_indexer(data.dates[earlier]).max())
    opens = int(position.get_indexer(data.dates[later]).min()) - creator.lookback + 1
    checks.append(
        {
            "boundary": f"{earlier} -> {later}",
            "last_label_row": last_label,
            "label_period_ends": last_label + creator.horizon,
            "next_window_opens": opens,
            "disjoint": last_label + creator.horizon < opens,
        }
    )

purge_check = pd.DataFrame(checks)
purge_check

,boundary,last_label_row,label_period_ends,next_window_opens,disjoint
0,train -> val,2936,2941,2942,True
1,val -> test,3570,3575,3576,True


## Coverage — what the train slice can actually see

The train figure is the one that matters. A channel at 20% overall could be 20%
everywhere, or 0% in train and 70% at test — and only the second is a problem.
Anything at `train_coverage = 0` was dropped above.

In [8]:
data.coverage

,coverage,train_coverage,train_nunique
prop_buy_vol,0.200473,0.000000,0
vietnam__economy__prices__economics__vncpit,0.607329,0.434457,61
vietnam__economy__prices__economics__vncir,0.652009,0.498808,52
vietnam__economy__prices__economics__vncirmm,0.652009,0.498808,34
european_union__economy__consumer__economics__euccr,0.669976,0.524685,75
...,...,...,...
usa__economy__prices__fred__wpusi019011,1.000000,1.000000,140
vietnam__economy__prices__economics__vnep,1.000000,1.000000,13
vietnam__economy__prices__economics__vnirmm,1.000000,1.000000,99
vietnam__economy__trade__economics__vnbot,1.000000,1.000000,141


## Drift — how much of the test set left the train range

The panel is non-stationary, so a train-fitted scaler necessarily maps part of
the test period outside the range the model was fitted on. This measures it per
channel instead of leaving it to be discovered from a bad test score.

In [9]:
summary = creator.drift_summary(data)
print(
    f"{summary['channels_over_1pct']} of {summary['scaled_channels']} channels put"
    f" >1% of TEST beyond {summary['sigma']} train-sigmas; "
    f"{summary['channels_fully_outside']} put ALL of it there."
)

48 of 202 channels put >1% of TEST beyond 5.0 train-sigmas; 4 put ALL of it there.


In [10]:
data.drift

,test_beyond_5sd,test_mean_z
russia__economy__gdp__economics__rugdppa,1.000000,9.120275
netherlands__economy__money__economics__nlfer,1.000000,9.434043
united_kingdom__economy__housing__economics__gbmr,1.000000,12.580196
netherlands__economy__trade__economics__nledtgdp,1.000000,-5.900557
usa__economy__government__fred__fyoint,0.955906,10.078051
...,...,...
usa__economy__prices__fred__will5000ind,0.000000,0.779682
vietnam__economy__money__economics__vnm1,0.000000,1.682221
vietnam__economy__prices__economics__vnep,0.000000,1.707211
vietnam__economy__prices__economics__vnirmm,0.000000,-0.187550


## Save

Writes the six tensors, the per-sample dates and tickers, both scalers,
`metadata.json`, `coverage.csv` and `drift.csv`.

⚠️ `replace=True` overwrites — any model run referencing the old dataset hash
will no longer verify.

In [11]:
directory = creator.save(data, replace=True)

listing = pd.DataFrame(
    [
        {
            "file": name,
            "MB": round(os.path.getsize(os.path.join(directory, name)) / 1e6, 3),
        }
        for name in sorted(os.listdir(directory))
    ]
)
print(directory)
listing

D:\GIT\master-thesis\src\train_test_set\vcb__return_5day__final__d20_h5__tr70_val15_test15__std


,file,MB
0,X_test.npy,10.262
1,X_train.npy,47.155
2,X_val.npy,9.858
3,coverage.csv,0.012
4,dates_test.npy,0.026
5,dates_train.npy,0.117
6,dates_val.npy,0.025
7,drift.csv,0.014
8,feature_scaler.pkl,0.005
9,metadata.json,0.025


## Load it back the way the model stage will

`model/common/data.py` loads these files **by name** and hashes the six tensors.
If this cell works, the dataset is model-ready.

In [12]:
from model.common.data import load_dataset

loaded = load_dataset(creator.name)
print(f"name     {loaded.name}")
print(f"hash     {loaded.hash}")
print(f"features {loaded.n_features}   lookback {loaded.lookback}")
print(f"X_train  {loaded.X_train.shape}")
print(f"test     {loaded.dates_test[0]} -> {loaded.dates_test[-1]}")
print(f"scalers  {type(loaded.feature_scaler).__name__} / {type(loaded.target_scaler).__name__}")

name     vcb__return_5day__final__d20_h5__tr70_val15_test15__std
hash     6f657cd4ea02ae5a
features 202   lookback 20
X_train  (2918, 20, 202)
test     2023-11-21 -> 2026-06-18
scalers  StandardScaler / StandardScaler


## ⚠️ What this dataset does not claim

All 19 source runs of this table computed **no null**
(`feature_selection/CONTEXT.md` §14b), and 199 of the 203 channels were chosen
by exactly one run (`final_features/CONTEXT.md` §6) — the table is a union of
unstable shortlists, not a consensus. These tensors reshape those channels.
They do not vouch for them.